In [13]:
import gurobipy as gp
from gurobipy import GRB

# Base data
commodities = ["Methane", "Hydrogen"]
nodes = ["Denmark", "Sweden", "Norway", "Finland", "Germany"]

# Arcs
arcs_base = [
    ("Denmark", "Norway"),
    ("Denmark", "Finland"),
    ("Denmark", "Germany"),
    ("Sweden", "Norway"),
    ("Sweden", "Finland"),
    ("Sweden", "Germany"),
]

# Reverse arcs
arcs_reverse = [(destination, source) for source, destination in arcs_base]

# Combined arcs
arcs = arcs_base + arcs_reverse

# Capacity for quadruples commodity-source-destination
capacity_base = {
    ("Methane", "Denmark", "Norway"): 100,
    ("Methane", "Denmark", "Finland"): 80,
    ("Methane", "Denmark", "Germany"): 120,
    ("Methane", "Sweden", "Norway"): 120,
    ("Methane", "Sweden", "Finland"): 120,
    ("Methane", "Sweden", "Germany"): 120,
    ("Hydrogen", "Denmark", "Norway"): 0,
    ("Hydrogen", "Denmark", "Finland"): 0,
    ("Hydrogen", "Denmark", "Germany"): 0,
    ("Hydrogen", "Sweden", "Norway"): 0,
    ("Hydrogen", "Sweden", "Finland"): 0,
    ("Hydrogen", "Sweden", "Germany"): 0,
}

# Reverse cap
capacity_reverse = {(k, j, i): cap for (k, i, j), cap in capacity_base.items()}

capacity = {**capacity_base, **capacity_reverse}

# Capacity for quadruples commodity-source-destination
capacity_limit_base = {
    ("Methane", "Denmark", "Norway"): 400,
    ("Methane", "Denmark", "Finland"): 100,
    ("Methane", "Denmark", "Germany"): 120,
    ("Methane", "Sweden", "Norway"): 120,
    ("Methane", "Sweden", "Finland"): 120,
    ("Methane", "Sweden", "Germany"): 120,
    ("Hydrogen", "Denmark", "Norway"): 150,
    ("Hydrogen", "Denmark", "Finland"): 120,
    ("Hydrogen", "Denmark", "Germany"): 100,
    ("Hydrogen", "Sweden", "Norway"): 100,
    ("Hydrogen", "Sweden", "Finland"): 150,
    ("Hydrogen", "Sweden", "Germany"): 80,
}

# Reverse cap
capacity_limit_reverse = {(k, j, i): cap for (k, i, j), cap in capacity_limit_base.items()}

capacity_limit = {**capacity_limit_base, **capacity_limit_reverse}

# Cost for triplets commodity-source-destination
cost_base = {
    ("Methane", "Denmark", "Norway"): 10,
    ("Methane", "Denmark", "Finland"): 20,
    ("Methane", "Denmark", "Germany"): 60,
    ("Methane", "Sweden", "Norway"): 40,
    ("Methane", "Sweden", "Finland"): 40,
    ("Methane", "Sweden", "Germany"): 30,
    ("Hydrogen", "Denmark", "Norway"): 20,
    ("Hydrogen", "Denmark", "Finland"): 20,
    ("Hydrogen", "Denmark", "Germany"): 80,
    ("Hydrogen", "Sweden", "Norway"): 60,
    ("Hydrogen", "Sweden", "Finland"): 70,
    ("Hydrogen", "Sweden", "Germany"): 30,
}

# Reverse cost
cost_reverse = {(k, j, i): cost for (k, i, j), cost in cost_base.items()}

cost = {**cost_base, **cost_reverse}

# Cost for triplets commodity-source-destination
investment_cost_base = {
    ("Methane", "Denmark", "Norway"): 10,
    ("Methane", "Denmark", "Finland"): 20,
    ("Methane", "Denmark", "Germany"): 60,
    ("Methane", "Sweden", "Norway"): 40,
    ("Methane", "Sweden", "Finland"): 40,
    ("Methane", "Sweden", "Germany"): 30,
    ("Hydrogen", "Denmark", "Norway"): 20,
    ("Hydrogen", "Denmark", "Finland"): 20,
    ("Hydrogen", "Denmark", "Germany"): 80,
    ("Hydrogen", "Sweden", "Norway"): 60,
    ("Hydrogen", "Sweden", "Finland"): 70,
    ("Hydrogen", "Sweden", "Germany"): 30,
}

# Reverse cost
investment_cost_reverse = {(k, j, i): cost for (k, i, j), cost in investment_cost_base.items()}

investment_cost = {**investment_cost_base, **investment_cost_reverse}

conversion_cost_base = {
    ("Methane", "Denmark", "Norway"): 10,
    ("Methane", "Denmark", "Finland"): 20,
    ("Methane", "Denmark", "Germany"): 60,
    ("Methane", "Sweden", "Norway"): 40,
    ("Methane", "Sweden", "Finland"): 40,
    ("Methane", "Sweden", "Germany"): 30,
    ("Hydrogen", "Denmark", "Norway"): 2,
    ("Hydrogen", "Denmark", "Finland"): 2,
    ("Hydrogen", "Denmark", "Germany"): 8,
    ("Hydrogen", "Sweden", "Norway"): 6,
    ("Hydrogen", "Sweden", "Finland"): 7,
    ("Hydrogen", "Sweden", "Germany"): 3,
}

# Reverse cost
conversion_cost_reverse = {(k, j, i): cost for (k, i, j), cost in conversion_cost_base.items()}

conversion_cost = {**conversion_cost_base, **conversion_cost_reverse}

# Supply (> 0) and demand (< 0) for pairs of commodity-city
inflow = {
    ("Methane", "Denmark"): -500,
    ("Methane", "Sweden"): 0,
    ("Methane", "Norway"): 500,
    ("Methane", "Finland"): 0,
    ("Methane", "Germany"): 0,
    ("Hydrogen", "Denmark"): 60,
    ("Hydrogen", "Sweden"): 40,
    ("Hydrogen", "Norway"): -40,
    ("Hydrogen", "Finland"): -30,
    ("Hydrogen", "Germany"): -30,
}

# Create optimization model
m = gp.Model("netflow")

# Create variables
flow = m.addVars(commodities, arcs, vtype=GRB.CONTINUOUS, name="flow")
# variable for capacity investment
investment = m.addVars(commodities, arcs, vtype=GRB.CONTINUOUS, name="investment")
# Variable for converted capacity
converted_capacity = m.addVars(commodities, arcs, vtype=GRB.CONTINUOUS, name="converted_capacity")
# Binary variable to indicate if conversion is active
conversion_indicator = m.addVars(commodities, arcs, vtype=GRB.BINARY, name="conversion_indicator")

# Set objective function
m.setObjective(
    gp.quicksum(flow[h, i, j] * cost[h, i, j] for h in commodities for i, j in arcs) +
    gp.quicksum(investment[h, i, j] * investment_cost[h, i, j] for h in commodities for i, j in arcs_base) +
    gp.quicksum(converted_capacity[h, i, j] * conversion_cost[h, j, i] for h in commodities for i, j in arcs_base),
    GRB.MINIMIZE
)

# Arc-capacity constraints with investment
m.addConstrs((flow[h, i, j] <= converted_capacity[h, i, j] + investment[h, i, j] for h in commodities for i, j in arcs), "cap")

# Arc-capacity total limit
m.addConstrs((investment[h, i, j] + converted_capacity[h, i, j] 
              <= capacity_limit[h, i, j] for h in commodities for i, j in arcs_base), "cap")

# Investment constraints
m.addConstrs((investment[h, i, j] == investment[h, j, i] for h in commodities for i, j in arcs), "conversion")

#conversion binary constraint
m.addConstrs((gp.quicksum(conversion_indicator[h, i, j] for h in commodities) == 1 for i, j in arcs), "conversion_indicator_sum_one")

#link the converted capacity to the initial capacity
m.addConstrs((capacity[commodities[0], i, j] * conversion_indicator[h, i, j] 
              == converted_capacity[h, i, j] for h in commodities[1:] for i, j in arcs), "capacity_relationship")
m.addConstrs((capacity[commodities[0], i, j] * conversion_indicator[commodities[0], i, j]
        == converted_capacity[commodities[0], i, j]
        for i, j in arcs), "capacity_relationship_first_commodity")

# Conversion constraints
m.addConstrs((converted_capacity[h, i, j] == converted_capacity[h, j, i] for h in commodities for i, j in arcs), "conversion")

# Flow-conservation constraints
m.addConstrs(
    (
        flow.sum(h, "*", j) + inflow[h, j] == flow.sum(h, j, "*")
        for h in commodities
        for j in nodes
    ),
    "node",
)

# Compute optimal solution
m.optimize()

if m.Status == GRB.OPTIMAL:
    solution = m.getAttr("X", flow)
    investment_solution = m.getAttr("X", investment)
    converted_capacity_solution = m.getAttr("X", converted_capacity)
    conversion_indicator_solution = m.getAttr("X", conversion_indicator)

    for h in commodities:
        print(f"\nOptimal flows for {h}:")
        for i, j in arcs:
            if solution[h, i, j] > 0:
                print(f"{i} -> {j}: {solution[h, i, j]:g}")

        print(f"\nOptimal investments for {h}:")
        for i, j in arcs_base:
            if investment_solution[h, i, j] > 0:
                print(f"Investment for {h} in edge ({i} -> {j}): {investment_solution[h, i, j]:g}")

        print(f"\nOptimal converted capacities for {h}:")
        for i, j in arcs_base:
            if converted_capacity_solution[h, i, j] > 0:
                print(f"Converted capacity for {h} from {i} to {j}: {converted_capacity_solution[h, i, j]:g}")

    print("\nOptimal conversion indicators:")
    for h in commodities:
        for i, j in arcs_base:
            #
                print(f"Conversion of {h} from {i} to {j} is: {conversion_indicator_solution[h, i, j]:g}")

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 130 rows, 96 columns and 312 nonzeros
Model fingerprint: 0x06daf585
Variable types: 72 continuous, 24 integer (24 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  Objective range  [2e+00, 8e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+02]
Presolve removed 113 rows and 77 columns
Presolve time: 0.00s
Presolved: 17 rows, 19 columns, 50 nonzeros
Variable types: 18 continuous, 1 integer (1 binary)
Found heuristic solution: objective 41500.000000

Root relaxation: objective 4.138000e+04, 7 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     